# Getting Started with LangGraph
## LangGraph: A Framework for Stateful, Graph-Based AI Workflows


This Jupyter Notebook is designed to help you understand key concepts in LangGraph, a powerful framework for building stateful, graph-based AI workflows using Large Language Models (LLMs). It is assumed you are familiar with LangChain, but LangGraph is new to you. The notebook is divided into three parts: Beginner, Intermediate, and Advanced, guiding you step-by-step from basic concepts to complex multi-agent systems.

## What is LangGraph?
LangGraph is a library built on top of LangChain that allows you to create stateful, graph-based workflows for AI applications. Unlike traditional linear chains in LangChain, LangGraph models interactions as a graph, where nodes represent actions (e.g., calling an LLM, using a tool) and edges define how to move between them. This is particularly useful for applications requiring state management, cycles, or complex decision-making.

Key Components:
- Nodes: Individual steps or actions (e.g., call an LLM, execute a tool).
- Edges: Connections between nodes, defining the flow (can be conditional).
- State: A shared data structure that persists across nodes, tracking information like user input, LLM outputs, or tool results.
- Graph: The overall structure combining nodes and edges.

Why LangGraph?:
- Enables stateful workflows where previous steps influence future ones.
- Supports cycles for iterative processes (e.g., retrying a tool if the answer is incorrect).
- Ideal for agentic systems where an AI decides which tools to use dynamically.

## When to Use LangGraph?
Use LangGraph when your application needs:
- Complex workflows: When a simple linear chain (e.g., LangChain’s Chain) isn’t enough.
- State persistence: To maintain context across multiple steps (e.g., user query history).
- Decision-making agents: For scenarios where an AI must choose tools or actions dynamically.
- Iterative processes: When you need loops, retries, or conditional branching (e.g., verify and retry if a tool’s output is wrong).

Examples:
- Building a chatbot that uses tools (e.g., search, calculator) based on user queries.
- Creating multi-agent systems where agents collaborate to solve a task.
- Managing workflows with conditional logic, like routing queries to different LLMs or tools.

## Implementing an AI Agent with Langgraph & Langchain

In [12]:
!python -m pip install pyowm wikipedia langchain-google-genai --quiet


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [13]:
# initializing the llm
model_name="gemini-2.0-flash"
from langchain.chat_models import init_chat_model
model = init_chat_model(model_name, model_provider="google_genai")


# load a tool
from langchain.utilities import WikipediaAPIWrapper
from langchain.tools import OpenWeatherMapQueryRun,WikipediaQueryRun

wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
#owm = OpenWeatherMapQueryRun() # make sure to have env variable for OPENWEATHERMAP_API_KEY

tools = [wiki]

# initialize agent
from langgraph.prebuilt import create_react_agent
agent = create_react_agent(model,tools)

In [14]:
from langchain_core.messages import HumanMessage
response = agent.invoke({"messages":HumanMessage(content="hi")})
response['messages']

[HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='2bdfb1a1-a635-4887-8584-f1272475b396'),
 AIMessage(content='Hi, how can I help you today?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--afe40315-9472-4f34-a13b-1effa7bf6094-0', usage_metadata={'input_tokens': 49, 'output_tokens': 10, 'total_tokens': 59, 'input_token_details': {'cache_read': 0}})]

In [15]:
response = agent.invoke({"messages":HumanMessage(content="Where is the city Kualalumpur located?")})
response['messages']

[HumanMessage(content='Where is the city Kualalumpur located?', additional_kwargs={}, response_metadata={}, id='ceb653dd-f4f3-4883-8d55-13adc32d1812'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'wikipedia', 'arguments': '{"query": "Kualalumpur location"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--efba211f-5d7a-401c-95a6-ec4dc535c34b-0', tool_calls=[{'name': 'wikipedia', 'args': {'query': 'Kualalumpur location'}, 'id': '0e1c1739-699e-430e-9f8c-5676d2da4961', 'type': 'tool_call'}], usage_metadata={'input_tokens': 59, 'output_tokens': 8, 'total_tokens': 67, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content="Page: Kuala Lumpur\nSummary: Kuala Lumpur (KL), officially the Federal Territory of Kuala Lumpur, is the capital city and a federal territory of Malaysia. It is the most populous city in the country, covering an ar

In [16]:
for step in agent.stream({"messages":[HumanMessage(content="Where is the city Kualalumpur located?")]},
                         stream_mode='values'):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Where is the city Kualalumpur located?
================================== Ai Message ==================================
Tool Calls:
  wikipedia (497feccb-69df-4b48-920d-37f5b255561d)
 Call ID: 497feccb-69df-4b48-920d-37f5b255561d
  Args:
    query: Kuala Lumpur location
================================= Tool Message =================================
Name: wikipedia

Page: Kuala Lumpur City F.C.
Summary: Kuala Lumpur City Football Club, known simply as KL City FC, is a Malaysian professional football club based in Kuala Lumpur. The club competes in the Malaysia Super League, the top level of Malaysian football, and was founded in 1974 as Federal Territory by the Kuala Lumpur Football Association (KLFA). It was later renamed Kuala Lumpur FA and Kuala Lumpur United, before renaming to its current name in 2021.
Kuala Lumpur City have won two Malaysian league titles, four Malaysia Cups, three Malaysia FA Cups,

In [17]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()



wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
owm = OpenWeatherMapQueryRun() # make sure to have env variable for OPENWEATHERMAP_API_KEY

tools = [wiki,owm]


agent = create_react_agent(model,tools,checkpointer=memory)

config = {"configurable":{"thread_id":"abc112"}}

for step in agent.stream({"messages":[HumanMessage(content="Where is the city Kualalumpur located?")]},
                         stream_mode='values',config=config):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Where is the city Kualalumpur located?
================================== Ai Message ==================================
Tool Calls:
  wikipedia (8ca82016-18be-4a8d-94ee-f3e2abecc6d5)
 Call ID: 8ca82016-18be-4a8d-94ee-f3e2abecc6d5
  Args:
    query: Kuala Lumpur location
================================= Tool Message =================================
Name: wikipedia

Page: Kuala Lumpur City F.C.
Summary: Kuala Lumpur City Football Club, known simply as KL City FC, is a Malaysian professional football club based in Kuala Lumpur. The club competes in the Malaysia Super League, the top level of Malaysian football, and was founded in 1974 as Federal Territory by the Kuala Lumpur Football Association (KLFA). It was later renamed Kuala Lumpur FA and Kuala Lumpur United, before renaming to its current name in 2021.
Kuala Lumpur City have won two Malaysian league titles, four Malaysia Cups, three Malaysia FA Cups,

In [18]:
for step in agent.stream({"messages":[HumanMessage(content="I live there.")]},
                         stream_mode='values',config=config):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

I live there.
================================== Ai Message ==================================

Ah, that's wonderful! It's a vibrant and exciting city. Is there anything specific you'd like to know or any way I can help you today? Perhaps you'd like to know about the weather there?


In [19]:
for step in agent.stream({"messages":[HumanMessage(content="Whats the weather there?.")]},
                         stream_mode='values',config=config):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Whats the weather there?.
================================== Ai Message ==================================
Tool Calls:
  open_weather_map (7be656de-a8bc-4980-bb8a-b3307dd8a800)
 Call ID: 7be656de-a8bc-4980-bb8a-b3307dd8a800
  Args:
    location: Kuala Lumpur, MY
================================= Tool Message =================================
Name: open_weather_map

In Kuala Lumpur, MY, the current weather is as follows:
Detailed status: overcast clouds
Wind speed: 2.41 m/s, direction: 247°
Humidity: 56%
Temperature: 
  - Current: 32.37°C
  - High: 32.37°C
  - Low: 32.37°C
  - Feels like: 36.71°C
Rain: {}
Heat index: None
Cloud cover: 89%
================================== Ai Message ==================================

The weather in Kuala Lumpur is currently 32.37°C, but it feels like 36.71°C. It's cloudy with a wind speed of 2.41 m/s. The humidity is 56%.
